In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Human and LLM agreement

Calculate the SI S2.2 tables from 200 human-rated triples, 800 additional tasks and model judgments. Run from `revision_notebooks/` after notebooks 01 and 02. Read their files and the human ratings `batch11.tsv` and `batch12.tsv` from `DATA/validation/`; save tables under `OUTPUTS/validation/`.

The human metric table uses tasks where both raters are decisive. Each model/rater comparison excludes that pair's “not sure” ratings. The score-based choice is Patent 1 for a positive score difference and Patent 2 otherwise, including tied or missing differences.


In [ ]:
import json
import math
import numpy as np
import pandas as pd

VALIDATION = DATA / 'validation'
OUT = OUTPUTS / 'validation'
OUT.mkdir(parents=True, exist_ok=True)
METRICS = ('abstract', 'summary', 'claim', 'detail')
TERCILES = ('Same tercile', '1 tercile diff', '2 tercile diff')
LOW_MID, MID_HIGH = 2.915233, 3.109160


In [ ]:
def load_tasks(path=VALIDATION / "tasks.jsonl"):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]

def category(score):
    return "high" if score > MID_HIGH else "mid" if score > LOW_MID else "low"

def agreement(left, right):
    """Exclude 'not sure' (0) separately for each pairwise comparison."""
    decisive = left.isin([1, 2]) & right.isin([1, 2])
    a, b = left[decisive], right[decisive]
    n, k = len(a), int((a == b).sum())
    return {"agreements": k, "n": n, "agreement": k / n if n else np.nan}

def metric_choice(data, metric):
    # Assign tied or missing score differences to Patent 2.
    difference = (data[f"patent_1_{metric}"] - data[f"patent_2_{metric}"]).fillna(0)
    return pd.Series(np.where(difference > 0, 1, 2), index=data.index)


## Join scores, ratings and saved judgments

Full-text judgments cover only the 200 human tasks. Left joins retain all 1,000 title/abstract tasks when adding that smaller judgment table.

In [ ]:
tasks = pd.DataFrame(load_tasks(VALIDATION / "tasks.jsonl"))
keys = ["focal_patent_id", "patent_1_id", "patent_2_id"]
data = tasks[["task_id", "source", *keys]].merge(
    pd.read_parquet(VALIDATION / "pair_scores.parquet"), on="task_id")
h1 = pd.read_csv(VALIDATION / "batch11.tsv", sep="\t")
h2 = pd.read_csv(VALIDATION / "batch12.tsv", sep="\t")
human = h1.merge(h2, on=["ccmt_patent_id", "patent_id1", "patent_id2"])
human = human.rename(columns=dict(zip(["ccmt_patent_id", "patent_id1", "patent_id2"], keys)))
data = data.merge(human, on=keys, how="left")
for model in ("deepseek", "glm", "deepseek_fulltext"):
    judged = pd.read_csv(VALIDATION / f"{model}_judgments.tsv", sep="\t")
    judged = judged[["task_id", "choice", "confidence"]].rename(
        columns={"choice": model, "confidence": f"{model}_confidence"})
    data = data.merge(judged, on="task_id", how="left")
category_number = {"low": 0, "mid": 1, "high": 2}
categories = [data[f"patent_{i}_score"].map(category).map(category_number) for i in (1, 2)]
data["tercile_distance"] = (categories[0] - categories[1]).abs()
data["consensus"] = data.choice1.where(data.choice1.eq(data.choice2) & data.choice1.isin([1, 2]))
data.source.value_counts()


## Human agreement and the normalized panel

Group composite scores using cutoffs 2.915233 and 3.109160. Calculate agreement and sample counts from the supplied ratings.


In [ ]:
human = data[data.source == "human"]
# Use the common sample where both raters are decisive.
human = human[human.choice1.isin([1, 2]) & human.choice2.isin([1, 2])]
rows, counts = [], []
for distance, label in [*enumerate(TERCILES), (None, "All")]:
    subset = human if distance is None else human[human.tercile_distance == distance]
    hh = agreement(subset.choice1, subset.choice2)
    row = {"Tercile": label, "Inter-rater": hh["agreement"]}
    counts.append({"sample": label, "comparison": "Human 1 vs Human 2", **hh})
    for metric, display in zip(METRICS, ("Abstract", "Summary", "Claims", "Detail")):
        prediction = metric_choice(subset, metric)
        rates = []
        for rater in (1, 2):
            result = agreement(prediction, subset[f"choice{rater}"])
            rates.append(result["agreement"])
            counts.append({"sample": label, "comparison": f"{display} vs Human {rater}", **result})
        row[display] = np.mean(rates)
    rows.append(row)
average = pd.DataFrame(rows).set_index("Tercile")
normalized = average.drop(columns="Inter-rater").div(average["Inter-rater"], axis=0)
human_counts = counts
average.round(3)


In [ ]:
normalized.round(3)


## Model agreement and full-text sensitivity

In [ ]:
human = data[data.source == "human"]
extra = data[data.source == "llm_extra"]
rows, counts, fulltext = {}, [], []
labels = ["Human rater 1", "Human rater 2", "Average across individual raters",
          "Agreement with both human raters", "Higher composite text-similarity score",
          "Higher abstract similarity"]
for model, display in (("deepseek", "DS"), ("glm", "GLM"), ("deepseek_fulltext", "DS full text")):
    rates = []
    for rater in (1, 2):
        result = agreement(human[model], human[f"choice{rater}"])
        counts.append({"sample": "200 human", "comparison": f"{display} vs Human {rater}", **result})
        rates.append(result["agreement"])
    rater_average = np.mean(rates)  # Average of two proportions, not pooled counts.
    fulltext.append({"Judge": display, "Average across individual raters": rater_average})
    if model == "deepseek_fulltext":
        continue
    consensus = agreement(human[model], human.consensus)
    counts.append({"sample": "142 consensus", "comparison": f"{display} vs human consensus", **consensus})
    values = [*rates, rater_average, consensus["agreement"]]
    for metric in ("score", "abstract"):
        result = agreement(extra[model], metric_choice(extra, metric))
        values.append(result["agreement"])
        counts.append({"sample": "800 extra", "comparison": f"{display} vs {metric}", **result})
    rows[display] = values
llm = pd.DataFrame(rows, index=pd.Index(labels, name="Comparison"))
fulltext = pd.DataFrame(fulltext)
llm_counts = counts
llm.round(3)


In [ ]:
fulltext.round(6)


## Agreement counts and intervals


In [ ]:
human = data[data.source == "human"]
consensus = human[human.consensus.isin([1, 2])]
disagreement = human[human.choice1.isin([1, 2]) & human.choice2.isin([1, 2]) & human.choice1.ne(human.choice2)]
rows = []
for label, subset in (("All 1000", data), ("Human consensus", consensus),
                      ("Human disagreement", disagreement)):
    result = agreement(subset.deepseek, subset.glm)
    rows.append({"sample": label, "comparison": "DS vs GLM", **result})
high = consensus[consensus.deepseek_confidence == 5]
rows.append({"sample": "DS confidence 5, human consensus", "comparison": "DS vs consensus",
             **agreement(high.deepseek, high.consensus)})
# Wilson interval and Cohen's kappa for agreement across all tasks.
n, p, z = rows[0]["n"], rows[0]["agreement"], 1.959963984540054
center = (p + z*z/(2*n)) / (1 + z*z/n)
width = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
joint = data[data.deepseek.isin([1, 2]) & data.glm.isin([1, 2])]
a, b = joint.deepseek.eq(1).mean(), joint.glm.eq(1).mean()
expected = a*b + (1-a)*(1-b)
rows[0].update(ci95_low=center-width, ci95_high=center+width, cohen_kappa=(p-expected)/(1-expected))
prose = pd.DataFrame(rows)
prose


## Write tables and denominators

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for name, table in (("human_agreement", average), ("human_normalized", normalized), ("llm_agreement", llm)):
    table.to_csv(OUT / f"{name}.csv")
    row_end = r" \\"
    lines = [r"\begin{tabular}{l" + "c" * len(table.columns) + "}", r"\toprule",
             " & ".join([table.index.name, *table.columns]) + row_end, r"\midrule"]
    lines += [" & ".join([label, *(f"{value:.3f}" for value in row)]) + row_end
              for label, row in table.iterrows()]
    lines += [r"\bottomrule", r"\end{tabular}"]
    (OUT / f"{name}.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"\n{name}\n{table.round(3).to_string()}")
fulltext.to_csv(OUT / "fulltext_human_agreement.csv", index=False)
pd.DataFrame(human_counts + llm_counts).to_csv(OUT / "agreement_counts.csv", index=False)
prose.to_csv(OUT / "retained_prose_counts.csv", index=False)
print("\nTask counts:", data.source.value_counts().to_dict())
print("\n" + fulltext.to_string(index=False))
print("\n" + prose.to_string(index=False))


The saved tables report human agreement, model agreement and the full-text comparison, with sample counts for each calculation. All three validation notebooks use `DATA/validation/`.
